In [1]:
import pandas as pd
import numpy as np
import plotly.express as px
import plotly.graph_objects as go
import plotly.io as pio


pio.templates.default = "plotly_white"

print("Plotly version:", __import__("plotly").__version__)

Plotly version: 7.0.0


## 1. Load and inspect the data

In [4]:
# Load the movie streaming dataset
df_raw = pd.read_csv("MoviesOnStreamingPlatforms.csv")

# Display dataset information
print("Dataset Shape:", df_raw.shape)

print("\nDataset Information:")
df_raw.info()

Dataset Shape: (9515, 11)

Dataset Information:
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 9515 entries, 0 to 9514
Data columns (total 11 columns):
 #   Column           Non-Null Count  Dtype 
---  ------           --------------  ----- 
 0   Unnamed: 0       9515 non-null   int64 
 1   ID               9515 non-null   int64 
 2   Title            9515 non-null   object
 3   Year             9515 non-null   int64 
 4   Age              5338 non-null   object
 5   Rotten Tomatoes  9508 non-null   object
 6   Netflix          9515 non-null   int64 
 7   Hulu             9515 non-null   int64 
 8   Prime Video      9515 non-null   int64 
 9   Disney+          9515 non-null   int64 
 10  Type             9515 non-null   int64 
dtypes: int64(8), object(3)
memory usage: 817.8+ KB


In [3]:
print("\nColumn Names:")
print(df_raw.columns.tolist())

print("\nFirst Five Rows:")
display(df_raw.head())


Column Names:
['Unnamed: 0', 'ID', 'Title', 'Year', 'Age', 'Rotten Tomatoes', 'Netflix', 'Hulu', 'Prime Video', 'Disney+', 'Type']

First Five Rows:


,Unnamed: 0,ID,Title,Year,Age,Rotten Tomatoes,Netflix,Hulu,Prime Video,Disney+,Type
0,0,1,The Irishman,2019,18+,98/100,1,0,0,0,0
1,1,2,Dangal,2016,7+,97/100,1,0,0,0,0
2,2,3,David Attenborough: A Life on Our Planet,2020,7+,95/100,1,0,0,0,0
3,3,4,Lagaan: Once Upon a Time in India,2001,7+,94/100,1,0,0,0,0
4,4,5,Roma,2018,18+,94/100,1,0,0,0,0


# Check duplicate records

In [5]:
# Check exact duplicate rows
exact_duplicates = df_raw.duplicated().sum()

print("Exact Duplicate Rows:", exact_duplicates)

# Check duplicate movies using Title and Year
key_duplicates = df_raw.duplicated(
    subset=["Title", "Year"]
).sum()

print("Duplicate Movies by Title + Year:", key_duplicates)

# Record original row count
rows_before = len(df_raw)

# Remove duplicate movie records using Title and Year
df = df_raw.drop_duplicates(
    subset=["Title", "Year"]
).copy()

# Remove the unused index column, if present
if "Unnamed: 0" in df.columns:
    df.drop(columns=["Unnamed: 0"], inplace=True)

# Display cleaning results
print("\nRows Before Cleaning:", rows_before)
print("Rows After Cleaning:", len(df))
print("Rows Removed:", rows_before - len(df))

print("\nAnalytical Dataset Shape:", df.shape)

display(df.head())

Exact Duplicate Rows: 0
Duplicate Movies by Title + Year: 0

Rows Before Cleaning: 9515
Rows After Cleaning: 9515
Rows Removed: 0

Analytical Dataset Shape: (9515, 10)


,ID,Title,Year,Age,Rotten Tomatoes,Netflix,Hulu,Prime Video,Disney+,Type
0,1,The Irishman,2019,18+,98/100,1,0,0,0,0
1,2,Dangal,2016,7+,97/100,1,0,0,0,0
2,3,David Attenborough: A Life on Our Planet,2020,7+,95/100,1,0,0,0,0
3,4,Lagaan: Once Upon a Time in India,2001,7+,94/100,1,0,0,0,0
4,5,Roma,2018,18+,94/100,1,0,0,0,0


## Analyse missing values and platform flags

In [6]:
missing_report = pd.DataFrame({
    "Column": df.columns,
    "Missing Count": df.isnull().sum().values,
    "Missing Percentage": (
        df.isnull().sum().values / len(df) * 100
    ).round(2)
})

missing_report = missing_report[
    missing_report["Missing Count"] > 0
].sort_values(
    by="Missing Count",
    ascending=False
)

print("Missing-Value Report:")

display(missing_report)


platform_cols = [
    "Netflix",
    "Hulu",
    "Prime Video",
    "Disney+"
]

print("\nPlatform Flag Analysis:")

for platform in platform_cols:

    values = pd.to_numeric(
        df[platform],
        errors="coerce"
    )

    zero_count = (values == 0).sum()
    one_count = (values == 1).sum()
    missing_count = values.isna().sum()

    print(f"\n{platform}")
    print("Available on platform (1):", one_count)
    print("Not available on platform (0):", zero_count)
    print("Missing flags:", missing_count)

Missing-Value Report:


,Column,Missing Count,Missing Percentage
3,Age,4177,43.90
4,Rotten Tomatoes,7,0.07



Platform Flag Analysis:

Netflix
Available on platform (1): 3695
Not available on platform (0): 5820
Missing flags: 0

Hulu
Available on platform (1): 1047
Not available on platform (0): 8468
Missing flags: 0

Prime Video
Available on platform (1): 4113
Not available on platform (0): 5402
Missing flags: 0

Disney+
Available on platform (1): 922
Not available on platform (0): 8593
Missing flags: 0


## Prepare the analysis columns

In [8]:
df["Year"] = pd.to_numeric(
    df["Year"],
    errors="coerce"
)

rt_text = df["Rotten Tomatoes"].astype("string").str.strip()

rt_numerator = pd.to_numeric(
    rt_text.str.extract(
        r"^\s*(\d+(?:\.\d+)?)\s*/\s*100\s*$",
        expand=False
    ),
    errors="coerce"
)

rt_numeric = pd.to_numeric(
    rt_text,
    errors="coerce"
)

rt_score = rt_numeric.copy()

needs_scaling = rt_score > 10

rt_score.loc[needs_scaling] = (
    rt_score.loc[needs_scaling] / 10
)


valid_fraction = rt_numerator.notna()

rt_score.loc[valid_fraction] = (
    rt_numerator.loc[valid_fraction] / 10
)

df["RT Score"] = rt_score

age_text = df["Age"].astype("string").str.strip()

age_text = age_text.mask(
    age_text.isna() | age_text.eq(""),
    "Missing/Unrated"
)

df["Age Display"] = age_text.astype(str)


for platform in platform_cols:

    df[platform] = pd.to_numeric(
        df[platform],
        errors="coerce"
    )


def get_available_platforms(row):

    available = [
        platform
        for platform in platform_cols
        if row[platform] == 1
    ]

    if available:
        return ", ".join(available)

    return "None marked"


df["Available Platforms"] = df.apply(
    get_available_platforms,
    axis=1
)


print("Valid Rotten Tomatoes Scores:",
      df["RT Score"].notna().sum())

print("Missing Rotten Tomatoes Scores:",
      df["RT Score"].isna().sum())

print("\nRating Summary:")

display(
    df["RT Score"].describe().to_frame(
        name="RT Score (0-10)"
    )
)

print("\nMean Rating:", round(df["RT Score"].mean(), 2))
print("Median Rating:", round(df["RT Score"].median(), 2))
print("Minimum Rating:", df["RT Score"].min())
print("Maximum Rating:", df["RT Score"].max())

print("\nPrepared Dataset:")

display(
    df[
        [
            "Title",
            "Year",
            "Age Display",
            "Rotten Tomatoes",
            "RT Score",
            "Available Platforms"
        ]
    ].head()
)

Valid Rotten Tomatoes Scores: 9508
Missing Rotten Tomatoes Scores: 7

Rating Summary:


,RT Score (0-10)
count,9508.0
mean,5.354501
std,1.319767
min,1.0
25%,4.4
50%,5.2
75%,6.2
max,9.8



Mean Rating: 5.35
Median Rating: 5.2
Minimum Rating: 1.0
Maximum Rating: 9.8

Prepared Dataset:


,Title,Year,Age Display,Rotten Tomatoes,RT Score,Available Platforms
0,The Irishman,2019,18+,98/100,9.8,Netflix
1,Dangal,2016,7+,97/100,9.7,Netflix
2,David Attenborough: A Life on Our Planet,2020,7+,95/100,9.5,Netflix
3,Lagaan: Once Upon a Time in India,2001,7+,94/100,9.4,Netflix
4,Roma,2018,18+,94/100,9.4,Netflix


## The interactive histogram with a range slider

In [9]:
rating_df = df.dropna(
    subset=["RT Score"]
).copy()

fig1 = px.histogram(
    rating_df,
    x="RT Score",
    nbins=20,
    title="Distribution of Rotten Tomatoes Movie Ratings",
    labels={
        "RT Score": "Rotten Tomatoes Score (0-10)",
        "count": "Number of Movies"
    },
    hover_data={
        "RT Score": ":.1f"
    },
    template="plotly_white"
)

fig1.update_traces(
    xbins=dict(
        start=0,
        end=10,
        size=0.5
    ),
    marker_line_width=0.5,
    opacity=0.85
)

fig1.update_xaxes(
    title_text="Rotten Tomatoes Score (0-10)",
    range=[0, 10],
    dtick=1,
    rangeslider=dict(
        visible=True,
        thickness=0.12
    )
)

fig1.update_yaxes(
    title_text="Number of Movies",
    rangemode="tozero"
)

fig1.update_layout(
    height=650,
    bargap=0.05,
    hovermode="x unified",
    title_x=0.02,
    margin=dict(
        l=60,
        r=30,
        t=80,
        b=40
    )
)

fig1.show()

## The interactive multivariate scatter plot

In [10]:
scatter_df = df.dropna(
    subset=["Year", "RT Score"]
).copy()

AGE_ORDER = [
    "all",
    "7+",
    "13+",
    "16+",
    "18+",
    "Missing/Unrated"
]


AGE_COLORS = {
    "all": "#16a085",
    "7+": "#e9c46a",
    "13+": "#f4a261",
    "16+": "#e76f51",
    "18+": "#6c5ce7",
    "Missing/Unrated": "#9ca3af"
}


fig2 = px.scatter(
    scatter_df,

    x="Year",

    y="RT Score",

    color="Age Display",

    category_orders={
        "Age Display": AGE_ORDER
    },

    color_discrete_map=AGE_COLORS,

    hover_name="Title",

    hover_data={
        "Year": ":.0f",
        "RT Score": ":.1f",
        "Rotten Tomatoes": True,
        "Age Display": True,
        "Available Platforms": True,
        "Netflix": True,
        "Hulu": True,
        "Prime Video": True,
        "Disney+": True
    },

    opacity=0.70,

    labels={
        "Year": "Movie Release Year",
        "RT Score": "Rotten Tomatoes Score (0-10)",
        "Age Display": "Age Classification"
    },

    title="Movie Release Year vs Rotten Tomatoes Score",

    template="plotly_white"
)


fig2.update_xaxes(
    title_text="Movie Release Year",
    dtick=10
)

fig2.update_yaxes(
    title_text="Rotten Tomatoes Score (0-10)",
    range=[0, 10],
    dtick=1
)


fig2.update_layout(
    height=700,
    hovermode="closest",
    title_x=0.02,
    legend_title_text="Age Classification",
    margin=dict(
        l=60,
        r=40,
        t=80,
        b=60
    )
)


fig2.show()

## Interactive Box Plot by Age Classification

In [ ]:
box_df = df.dropna(
    subset=["RT Score"]
).copy()

age_order = [
    "all",
    "7+",
    "13+",
    "16+",
    "18+",
    "Missing/Unrated"
]

age_order = [
    age for age in age_order
    if age in box_df["Age Display"].unique()
]

fig3 = px.box(
    box_df,
    x="Age Display",
    y="RT Score",
    color="Age Display",
    category_orders={
        "Age Display": age_order
    },
    points="outliers",
    hover_data={
        "Title": True,
        "Year": True,
        "RT Score": ":.2f"
    },
    title="Rotten Tomatoes Rating Distribution by Age Classification",
    labels={
        "Age Display": "Age Classification",
        "RT Score": "Rotten Tomatoes Score (0–10)"
    },
    template="plotly_white"
)

fig3.update_layout(
    height=600,
    showlegend=False,
    title_x=0.02,
    xaxis_title="Age Classification",
    yaxis_title="Rotten Tomatoes Score (0–10)"
)

fig3.show()

## Interactive Violin Plot by Age Classification

In [12]:
violin_df = df.dropna(
    subset=["RT Score"]
).copy()

age_order = [
    age for age in [
        "all",
        "7+",
        "13+",
        "16+",
        "18+",
        "Missing/Unrated"
    ]
    if age in violin_df["Age Display"].unique()
]

fig4 = px.violin(
    violin_df,
    x="Age Display",
    y="RT Score",
    color="Age Display",
    category_orders={
        "Age Display": age_order
    },
    box=True,
    points=False,
    hover_data={
        "Title": True,
        "Year": True,
        "RT Score": ":.2f"
    },
    title="Rotten Tomatoes Rating Density by Age Classification",
    labels={
        "Age Display": "Age Classification",
        "RT Score": "Rotten Tomatoes Score (0–10)"
    },
    template="plotly_white"
)

fig4.update_layout(
    height=600,
    showlegend=False,
    title_x=0.02,
    xaxis_title="Age Classification",
    yaxis_title="Rotten Tomatoes Score (0–10)"
)

fig4.update_yaxes(
    range=[0, 10],
    dtick=1
)

fig4.show()

## Interactive Correlation Heatmap between Year, RT Score, Netflix, Hulu, Prime Video, Disney+

In [14]:
correlation_columns = [
    "Year",
    "RT Score",
    "Netflix",
    "Hulu",
    "Prime Video",
    "Disney+"
]

correlation_data = df[correlation_columns].apply(
    pd.to_numeric,
    errors="coerce"
)

correlation_matrix = correlation_data.corr(
    method="pearson"
)

annotations = (
    correlation_matrix
    .round(2)
    .astype(str)
    .values
)

fig5 = go.Figure(
    data=go.Heatmap(
        z=correlation_matrix.values,
        x=correlation_matrix.columns,
        y=correlation_matrix.index,
        zmin=-1,
        zmax=1,
        zmid=0,
        colorscale="RdBu_r",
        text=annotations,
        texttemplate="%{text}",
        colorbar={
            "title": "Correlation"
        },
        hovertemplate=(
            "Variable 1: %{x}<br>"
            "Variable 2: %{y}<br>"
            "Pearson Correlation: %{z:.3f}"
            "<extra></extra>"
        )
    )
)

fig5.update_layout(
    title="Correlation Heatmap of Movie Streaming Data",
    title_x=0.02,
    height=650,
    xaxis_title="Variables",
    yaxis_title="Variables",
    template="plotly_white"
)

fig5.show()

display(correlation_matrix.round(3))

,Year,RT Score,Netflix,Hulu,Prime Video,Disney+
Year,1.000,0.036,0.307,0.080,-0.244,-0.178
RT Score,0.036,1.000,0.054,0.183,-0.208,0.118
Netflix,0.307,0.054,1.000,-0.253,-0.648,-0.257
Hulu,0.080,0.183,-0.253,1.000,-0.240,-0.108
Prime Video,-0.244,-0.208,-0.648,-0.240,1.000,-0.276
Disney+,-0.178,0.118,-0.257,-0.108,-0.276,1.000


## Interactive Average Rotten Tomatoes Rating by Year

In [15]:
yearly_df = df.dropna(
    subset=["Year", "RT Score"]
).copy()

yearly_rating = (
    yearly_df
    .groupby("Year", as_index=False)
    .agg(
        Average_RT_Score=("RT Score", "mean"),
        Movie_Count=("RT Score", "count")
    )
    .sort_values("Year")
)

fig6 = px.line(
    yearly_rating,
    x="Year",
    y="Average_RT_Score",
    markers=True,
    custom_data=["Movie_Count"],
    title="Average Rotten Tomatoes Rating by Release Year",
    labels={
        "Year": "Movie Release Year",
        "Average_RT_Score": "Average Rotten Tomatoes Score (0–10)"
    },
    template="plotly_white"
)

fig6.update_traces(
    line=dict(width=2),
    marker=dict(size=5),
    hovertemplate=(
        "Release Year: %{x:.0f}<br>"
        "Average Rating: %{y:.2f}/10<br>"
        "Movies Rated: %{customdata[0]}"
        "<extra></extra>"
    )
)

fig6.update_layout(
    height=600,
    title_x=0.02,
    xaxis_title="Movie Release Year",
    yaxis_title="Average Rotten Tomatoes Score (0–10)",
    hovermode="x unified"
)

fig6.update_yaxes(
    range=[0, 10],
    dtick=1
)

fig6.show()
display(yearly_rating.head(10))

,Year,Average_RT_Score,Movie_Count
0,1914,4.5,1
1,1915,4.8,2
2,1916,4.4,1
3,1919,6.2,1
4,1920,5.4,2
5,1922,5.8,1
6,1923,6.0,1
7,1925,5.76,5
8,1927,5.7,1
9,1928,5.45,2


## Interactive DONUT CHART: MOVIES BY NUMBER OF PLATFORMS

In [17]:
platform_cols = [
    "Netflix",
    "Hulu",
    "Prime Video",
    "Disney+"
]

platform_data = df[platform_cols].apply(
    pd.to_numeric,
    errors="coerce"
).fillna(0)

platform_data["Platform Count"] = platform_data.sum(axis=1)

overlap_counts = (
    platform_data["Platform Count"]
    .value_counts()
    .reindex([1, 2, 3, 4], fill_value=0)
    .sort_index()
)

platform_labels = [
    "1 Platform",
    "2 Platforms",
    "3 Platforms",
    "4 Platforms"
]

fig7 = go.Figure(
    data=[
        go.Pie(
            labels=platform_labels,
            values=overlap_counts.values,
            hole=0.58,
            sort=False,
            textinfo="label+percent",
            hovertemplate=(
                "<b>%{label}</b><br>"
                "Number of Movies: %{value:,}<br>"
                "Percentage: %{percent}"
                "<extra></extra>"
            )
        )
    ]
)

fig7.update_layout(
    title={
        "text": "Movies by Number of Streaming Platforms",
        "x": 0.02
    },
    template="plotly_white",
    height=600,
    showlegend=True,
    legend_title_text="Platform Availability",
    annotations=[
        dict(
            text=f"<b>{len(df):,}</b><br>Movies",
            x=0.5,
            y=0.5,
            font_size=20,
            showarrow=False
        )
    ],
    margin=dict(
        l=40,
        r=40,
        t=80,
        b=40
    )
)

fig7.show()

overlap_summary = pd.DataFrame({
    "Number of Platforms": platform_labels,
    "Number of Movies": overlap_counts.values
})

overlap_summary["Percentage"] = (
    overlap_summary["Number of Movies"]
    / overlap_summary["Number of Movies"].sum()
    * 100
).round(2)

display(overlap_summary)

,Number of Platforms,Number of Movies,Percentage
0,1 Platform,9262,97.34
1,2 Platforms,245,2.57
2,3 Platforms,7,0.07
3,4 Platforms,1,0.01
